#### Environment Setup & Raw Data Ingestion

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("Loading datasets...")
# Make sure these paths are perfectly correct for your computer
all_data_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\MBP ControllerData 0521760 Overlock.xlsx"
breakdown_file = r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\Synthetic Overlock Breakdowns.xlsx"

df_all = pd.read_excel(all_data_file)
df_breakdown = pd.read_excel(breakdown_file)

# Handle column spelling differences
col_healthy = 'Breakdown' if 'Breakdown' in df_all.columns else 'Brekadown'
col_broken = 'Breakdown' if 'Breakdown' in df_breakdown.columns else 'Brekadown'

# Isolate the healthy data 
df_healthy = df_all[df_all[col_healthy].isna()].copy()
print(f"Loaded {len(df_healthy)} healthy records and {len(df_breakdown)} breakdown records.")

Loading datasets...
Loaded 1604 healthy records and 704 breakdown records.


#### Feature Engineering & Data Fusion

In [2]:
print("Combining Vibration, Electrical, and Mechanical Data...")

def parse_vibration_to_bands(vib_str):
    if pd.isna(vib_str): return {}
    parts = str(vib_str).split(',')
    res = {}
    try:
        for i in range(0, len(parts)-1, 2):
            f_start = int(parts[i])
            f_end = f_start + 10
            res[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
    except Exception:
        pass
    return res

# The 10 extra features to track alongside vibration
extra_features = [
    'machineRPM', 
    'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
    'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax',
    'machinePowerMean', 'machinePowerMin', 'machinePowerMax'
]

# Process Healthy Data
h_vibs = pd.DataFrame(df_healthy['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
h_extras = df_healthy[extra_features].reset_index(drop=True)
df_h_ml = pd.concat([h_extras, h_vibs], axis=1)
df_h_ml['Label'] = 'Healthy Baseline'

# Process Breakdown Data
b_vibs = pd.DataFrame(df_breakdown['machineVibration'].apply(parse_vibration_to_bands).tolist()).fillna(0)
b_extras = df_breakdown[extra_features].reset_index(drop=True)
df_b_ml = pd.concat([b_extras, b_vibs], axis=1)
df_b_ml['Label'] = df_breakdown[col_broken].values

# Combine into one Master Dataset
df_ml_master = pd.concat([df_h_ml, df_b_ml]).fillna(0)
print(f"Master Dataset created! The AI will look at {df_ml_master.shape[1] - 1} different variables per machine.")

Combining Vibration, Electrical, and Mechanical Data...
Master Dataset created! The AI will look at 70 different variables per machine.


In [7]:
df_ml_master

,machineRPM,machineVoltageMean,machineVoltageMin,machineVoltageMax,machineCurrentMean,machineCurrentMin,machineCurrentMax,machinePowerMean,machinePowerMin,machinePowerMax,...,520-530Hz,530-540Hz,540-550Hz,550-560Hz,560-570Hz,570-580Hz,580-590Hz,590-600Hz,600-610Hz,Label
0,0,230.086883,229.895752,230.693634,0.898381,0.214838,1.060949,206.705628,49.390401,244.754145,...,61.0,113.0,135.0,84.0,242.0,197.0,184.0,98.0,156.0,Healthy Baseline
1,0,230.913208,230.454040,231.143677,0.711900,0.348818,1.281300,164.387029,80.386408,296.164354,...,239.0,425.0,441.0,620.0,290.0,309.0,360.0,410.0,160.0,Healthy Baseline
2,0,229.745468,228.830124,231.270386,2.008029,0.189137,3.949287,461.335668,43.280348,913.353113,...,655.0,646.0,872.0,484.0,581.0,377.0,448.0,639.0,304.0,Healthy Baseline
3,0,230.075165,230.017960,230.532852,0.411496,0.357294,0.418272,94.675095,82.183953,96.425373,...,240.0,375.0,240.0,448.0,67.0,102.0,69.0,30.0,114.0,Healthy Baseline
4,0,229.881470,228.927948,230.967972,2.823194,0.381439,4.738139,648.999992,87.321959,1094.358280,...,1739.0,974.0,725.0,614.0,1250.0,768.0,833.0,685.0,398.0,Healthy Baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699,0,228.493489,228.962006,230.605133,1.812518,0.223728,2.797531,414.680996,51.225294,645.125038,...,3219.0,2834.0,1999.0,2192.0,1374.0,1928.0,1370.0,2033.0,1989.0,Blade Broken
700,0,230.138596,229.516754,231.712189,2.326992,0.171204,3.570531,535.530624,39.294111,827.335583,...,804.0,1075.0,677.0,512.0,424.0,419.0,474.0,528.0,390.0,Waveness
701,0,228.550354,228.039185,229.537399,2.038338,0.284831,2.580100,465.862859,64.952551,592.229348,...,777.0,1054.0,917.0,553.0,753.0,673.0,670.0,598.0,476.0,High Foot Pressure
702,0,229.502853,228.700745,230.454437,1.788824,0.364561,3.261943,410.540231,83.375411,751.729207,...,571.0,616.0,470.0,529.0,435.0,376.0,435.0,883.0,343.0,Skip Stitches/Slip


#### Label Encoding & Feature Standardization

In [3]:
print("Formatting Data for Deep Learning...")

# Separate numbers (X) from labels (Y)
X_raw = df_ml_master.drop(columns=['Label']).values
y_text = df_ml_master['Label'].values

# Encode text labels into numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_text)
y_categorical = to_categorical(y_encoded)

# Scale the data so RPM (4000) and Current (1.5) are treated equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save the Scaler and Encoder to your hard drive
with open('sewing_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('sewing_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)
    
print("Translators saved successfully.")

Formatting Data for Deep Learning...
Translators saved successfully.


#### Time-Series Transformation (Sliding Window Generation)

In [4]:
print("Creating Sliding Time Windows for Predictive Maintenance...")

def create_sequences(data_X, data_Y, time_steps=5):
    X_seq, y_seq = [], []
    for i in range(len(data_X) - time_steps):
        # Grab a block of past data (e.g., 5 consecutive readings)
        X_seq.append(data_X[i : (i + time_steps)])
        # Grab the target label from the FUTURE
        y_seq.append(data_Y[i + time_steps]) 
    return np.array(X_seq), np.array(y_seq)

# Set how many past readings the AI should look at (Window Size)
TIME_STEPS = 5 
X_lstm, y_lstm = create_sequences(X_scaled, y_categorical, time_steps=TIME_STEPS)

# Split data: shuffle=False is CRITICAL to keep the timeline chronological!
X_train, X_test, y_train, y_test = train_test_split(X_lstm, y_lstm, test_size=0.2, random_state=42, shuffle=True)

print(f"Corrected LSTM Input Shape: {X_train.shape}")
print(f"The AI will look at {X_train.shape[1]} past records to predict the future.")

Creating Sliding Time Windows for Predictive Maintenance...
Corrected LSTM Input Shape: (1842, 5, 70)
The AI will look at 5 past records to predict the future.


#### Deep Learning Architecture Definition

In [5]:
print("Building the Predictive LSTM Brain...")

model = Sequential()

# First Layer: Reads the sliding time window
model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))

# Second Layer: Deep pattern extraction
model.add(LSTM(32, return_sequences=False))
model.add(Dropout(0.2))

# Dense processing
model.add(Dense(32, activation='relu'))

# Output Layer: Predicts the exact breakdown category
num_classes = y_categorical.shape[1]
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Building the Predictive LSTM Brain...


c:\Users\deela\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 5, 64)          │        34,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,560 (189.69 KB)

 Trainable params: 48,560 (189.69 KB)

 Non-trainable params: 0 (0.00 B)

#### Model Training, Evaluation, & Export

In [6]:
print("Commencing Model Training...")

history = model.fit(
    X_train, y_train,
    epochs=50,           
    batch_size=16,       
    validation_data=(X_test, y_test), 
    verbose=1
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("\n" + "="*40)
print(f"FINAL PREDICTIVE TEST ACCURACY: {accuracy * 100:.2f}%")
print("="*40)

model.save("sewing_machine_predictive_lstm.h5")
print("✅ SUCCESS! Saved the .h5 model to your computer.")

Commencing Model Training...
Epoch 1/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6056 - loss: 1.9544 - val_accuracy: 0.7158 - val_loss: 0.9482
Epoch 2/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7273 - loss: 0.9775 - val_accuracy: 0.7831 - val_loss: 0.7728
Epoch 3/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7946 - loss: 0.7449 - val_accuracy: 0.8416 - val_loss: 0.5857
Epoch 4/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8622 - loss: 0.5125 - val_accuracy: 0.8829 - val_loss: 0.4285
Epoch 5/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9051 - loss: 0.3619 - val_accuracy: 0.9067 - val_loss: 0.3654
Epoch 6/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9486 - loss: 0.2334 - val_accuracy: 0.9479 - val_loss: 0.2383
Epoch 7/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9613 - loss: 0.1850 - val_accuracy: 0.9414 - val_loss: 0.2185
Epoch 8/50
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9615 - l


FINAL PREDICTIVE TEST ACCURACY: 98.26%
✅ SUCCESS! Saved the .h5 model to your computer.
